# Day 16 — Basic Retrieval-Augmented Generation (RAG)

**Linkific AI/ML Internship — Month 1 Training**  
**Intern:** Shri Sanjaykumar V  
**Role:** AI/ML Intern  
**Organization:** Linkific  
**Date:** 17 September 2026  

---

### Task Overview & Objective:
Construct an end-to-end **Retrieval-Augmented Generation (RAG)** pipeline connecting synthetic demonstration company documentation to a language model. The pipeline dynamically processes user queries, executes dense vector similarity search over chunked document embeddings, retrieves top relevant context, and generates context-grounded answers. In addition, we empirically evaluate three chunk sizes (200, 400, and 800 characters) across five benchmark questions to **identify the best-performing chunk-size configuration for the selected demonstration dataset and evaluation questions**.


## 2. Executive Summary & Confidentiality Notice

> **Confidentiality Notice:** To strictly comply with corporate information security standards, **no real proprietary Linkific internal documents, employee records, private repositories, or credentials are used in this implementation**.  
> All documents utilized are **synthetic demonstration documents** created specifically for learning and experimenting with Retrieval-Augmented Generation.

### Key Deliverables:
1. **Synthetic Documentation Corpus:** 5 demonstration company policy and workflow documents (~10.6K characters).
2. **Dense Vector Embeddings:** Generated via Hugging Face `sentence-transformers/all-MiniLM-L6-v2` (384-dimensional dense vectors).
3. **Dual Vector Database Implementation:**
   - **ChromaDB:** Collection-based storage with metadata filtering and HNSW indexing.
   - **FAISS:** Low-level Inner Product similarity search (`IndexFlatIP` with L2-normalized vectors).
4. **Context-Grounded Answer Generation:** Local inference via Hugging Face `google-t5/t5-small` sequence-to-sequence transformer.
5. **Empirical Chunk Size Evaluation:** Transparent, deterministic rule-based evaluation comparing **200, 400, and 800 characters** across Relevance, Correctness, Completeness, and Grounding.


## 3. RAG Architecture & Core Concepts

### What is Retrieval-Augmented Generation (RAG)?
Retrieval-Augmented Generation is an AI architectural pattern that enhances language model generation by dynamically retrieving relevant facts from an external knowledge store and passing them as context inside the model's prompt.

**RAG provides retrieved source context to the language model, which can help ground responses in the available documentation.**

### End-to-End Architecture Workflow:
```text
  [ Raw Company Documents (5 Text Files) ]
                    │
                    ▼
  [ Document Chunking (200 / 400 / 800 chars) ]
                    │
                    ▼
  [ Embedding Generation (all-MiniLM-L6-v2, 384-dim) ]
                    │
                    ▼
  [ Vector Indexing (ChromaDB Collection / FAISS Index) ]
                    │
        User Query ──┴──► [ Semantic Search / Vector Similarity ]
                                    │
                                    ▼
                      [ Top-K Retrieved Context Chunks ]
                                    │
                                    ▼
            [ Prompt Construction: Context + Question ]
                                    │
                                    ▼
                    [ Language Model (T5 Seq2Seq LM) ]
                                    │
                                    ▼
                     [ Context-Grounded Final Answer ]
```


## 4. Imports and Environment Setup

We initialize the environment, configure logging, and import the core libraries:
- `sentence_transformers`: For generating dense 384-dimensional semantic embeddings.
- `chromadb`: For persistent and in-memory vector storage with metadata filtering.
- `faiss`: For high-efficiency Euclidean and Cosine vector similarity search.
- `transformers` & `torch`: For running local sequence-to-sequence answer generation.
- `pandas` & `numpy`: For structured data manipulation and metric tracking.
- `matplotlib`: For visual evaluation charts and workflow diagrams.


In [1]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd

# Suppress minor library warnings for clean notebook presentation
warnings.filterwarnings("ignore")
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import matplotlib.pyplot as plt
import chromadb
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Libraries imported successfully:")
print(f"  • ChromaDB Version:            {chromadb.__version__}")
print(f"  • FAISS Version:               {faiss.__version__}")
print(f"  • SentenceTransformers:        {SentenceTransformer.__module__}")
print(f"  • PyTorch Version:             {AutoTokenizer.__module__}")


Libraries imported successfully:
  • ChromaDB Version:            1.5.9
  • FAISS Version:               1.15.1
  • SentenceTransformers:        sentence_transformers.sentence_transformer.model
  • PyTorch Version:             transformers.models.auto.tokenization_auto


## 5. Loading Documents

To demonstrate RAG without compromising private company data, we load five synthetic demonstration documents created specifically for this internship module:
1. `onboarding.txt`: Employee and intern onboarding guidelines, workstation setup, and orientation checklist.
2. `leave_policy.txt`: Daily attendance, working hours, planned leave notices, and medical leave procedures.
3. `training_guidelines.txt`: Four-week curriculum roadmap, daily training schedule, and code quality standards.
4. `project_workflow.txt`: 5-stage project development lifecycle, pre-submission checklist, and mentor review protocols.
5. `submission_guidelines.txt`: Daily 6:00 PM deadline, required deliverables structure, Git hygiene, and tracker logging.


In [2]:
docs_dir = os.path.join(os.getcwd(), "documents")
documents = {}
doc_metadata = {}

for fname in sorted(os.listdir(docs_dir)):
    if fname.endswith(".txt"):
        fpath = os.path.join(docs_dir, fname)
        with open(fpath, "r", encoding="utf-8") as f:
            content = f.read()
        documents[fname] = content
        doc_metadata[fname] = {
            "source": fname,
            "char_count": len(content),
            "word_count": len(content.split()),
            "line_count": len(content.splitlines())
        }

meta_df = pd.DataFrame(doc_metadata.values())
display(meta_df)
print(f"Total Corpus: {meta_df['char_count'].sum()} characters across {len(documents)} documents.")


                      source  char_count  word_count  line_count
0           leave_policy.txt        1905         256          16
1             onboarding.txt        2128         262          21
2       project_workflow.txt        2102         259          23
3  submission_guidelines.txt        2236         280          24
4    training_guidelines.txt        2286         288          19
Total Corpus: 10657 characters across 5 documents.


## 6. Document Inspection

We inspect a snippet from `onboarding.txt` to verify document formatting, heading structures, and procedural guidelines.


In [3]:
sample_doc = "onboarding.txt"
print(f"--- PREVIEW: {sample_doc} ---")
print(documents[sample_doc][:420])
print("\n[... document continues ...]")


--- PREVIEW: onboarding.txt ---
DOCUMENT: EMPLOYEE & INTERN ONBOARDING GUIDE
NOTICE: Synthetic demonstration documents created for RAG experimentation.

1. Welcome and Orientation:
Welcome to the Linkific AI/ML Internship Training Program. All incoming interns participate in a structured o

[... document continues ...]


## 7. Document Chunking

Text documents cannot simply be passed in their entirety to an embedding model or LLM context window. We implement sliding-window chunking with configurable chunk sizes and overlap windows:
- **Chunk Size:** Maximum character count in a single chunk.
- **Overlap:** Character overlap between consecutive chunks to prevent breaking semantic thoughts across chunk boundaries.


In [4]:
def chunk_text(text, chunk_size, overlap, source_name):
    chunks = []
    start = 0
    idx = 0
    text_len = len(text)
    step = chunk_size - overlap
    
    while start < text_len:
        end = min(start + chunk_size, text_len)
        chunk_str = text[start:end].strip()
        if len(chunk_str) > 20:
            chunks.append({
                "id": f"{source_name}_chunk_{idx}",
                "text": chunk_str,
                "metadata": {
                    "source": source_name,
                    "chunk_id": idx,
                    "char_start": start,
                    "char_end": end,
                    "chunk_size": len(chunk_str)
                }
            })
            idx += 1
        if end >= text_len:
            break
        start += step
    return chunks

# Test chunking on onboarding.txt
sample_chunks_400 = chunk_text(documents["onboarding.txt"], chunk_size=400, overlap=50, source_name="onboarding.txt")
print(f"Generated {len(sample_chunks_400)} chunks for onboarding.txt at size 400.")
print("Chunk 0 Preview:\n", sample_chunks_400[0]["text"][:160], "...")


Generated 6 chunks for onboarding.txt at size 400.
Chunk 0 Preview:
DOCUMENT: EMPLOYEE & INTERN ONBOARDING GUIDE
NOTICE: Synthetic demonstration do ...


## 8. Embedding Generation

We load the pretrained `sentence-transformers/all-MiniLM-L6-v2` model from Hugging Face:
- Produces **384-dimensional dense vectors**.
- Maps semantically similar sentences to nearby geometric coordinates in vector space.
- Normalizes vectors to unit length ($L_2 = 1.0$), allowing Cosine Similarity to be computed via fast Inner Products.


In [5]:
embed_model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading embedding model: {embed_model_name}...")
embedder = SentenceTransformer(embed_model_name)
embed_dim = embedder.get_sentence_embedding_dimension()
print(f"Embedding Model Loaded successfully. Dimension: {embed_dim} dimensions.")

# Demonstrate semantic similarity
s1 = "Interns must submit daily project code by 6:00 PM."
s2 = "The task submission deadline is 18:00 every evening."
s3 = "Pizza recipe includes tomato sauce and mozzarella cheese."

embs = embedder.encode([s1, s2, s3], normalize_embeddings=True)
sim_related = np.dot(embs[0], embs[1])
sim_unrelated = np.dot(embs[0], embs[2])

print(f"Semantic Cosine Similarity (s1 vs s2 - Synonymous Deadline): {sim_related:.4f}")
print(f"Semantic Cosine Similarity (s1 vs s3 - Completely Unrelated):  {sim_unrelated:.4f}")


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2...
Embedding Model Loaded successfully. Dimension: 384 dimensions.
Semantic Cosine Similarity (s1 vs s2 - Synonymous Deadline): 0.5672
Semantic Cosine Similarity (s1 vs s3 - Completely Unrelated):  0.1022


## 9. ChromaDB Vector Store

We initialize ChromaDB as an in-memory vector database and create three separate collections corresponding to our test chunk sizes:
- `rag_chunks_200`
- `rag_chunks_400`
- `rag_chunks_800`


In [6]:
chroma_client = chromadb.Client()
chunk_configs = [
    {"size": 200, "overlap": 50, "col": "rag_chunks_200"},
    {"size": 400, "overlap": 50, "col": "rag_chunks_400"},
    {"size": 800, "overlap": 100, "col": "rag_chunks_800"}
]

chroma_collections = {}

for cfg in chunk_configs:
    c_size = cfg["size"]
    c_col_name = cfg["col"]
    overlap = cfg["overlap"]
    
    all_chunks = []
    for doc_name, doc_text in documents.items():
        all_chunks.extend(chunk_text(doc_text, c_size, overlap, doc_name))
        
    texts = [c["text"] for c in all_chunks]
    metas = [c["metadata"] for c in all_chunks]
    ids = [c["id"] for c in all_chunks]
    
    embeddings = embedder.encode(texts, normalize_embeddings=True).tolist()
    
    try:
        chroma_client.delete_collection(c_col_name)
    except Exception:
        pass
        
    col = chroma_client.create_collection(name=c_col_name, metadata={"hnsw:space": "cosine"})
    col.add(documents=texts, embeddings=embeddings, metadatas=metas, ids=ids)
    chroma_collections[c_size] = col
    print(f"ChromaDB Collection '{c_col_name}' populated with {len(all_chunks)} chunks.")


ChromaDB Collection 'rag_chunks_200' populated with 71 chunks.
ChromaDB Collection 'rag_chunks_400' populated with 32 chunks.
ChromaDB Collection 'rag_chunks_800' populated with 17 chunks.


## 10. FAISS Vector Store

Alongside ChromaDB, we build a **FAISS (Facebook AI Similarity Search)** index:
- Uses `faiss.IndexFlatIP` (Exact Inner Product) on unit-normalized vectors, which computes exact Cosine Similarity.
- Demonstrates how low-level vector libraries operate with matrix arrays.


In [7]:
faiss_indexes = {}
faiss_stores = {}

for cfg in chunk_configs:
    c_size = cfg["size"]
    overlap = cfg["overlap"]
    
    all_chunks = []
    for doc_name, doc_text in documents.items():
        all_chunks.extend(chunk_text(doc_text, c_size, overlap, doc_name))
        
    texts = [c["text"] for c in all_chunks]
    metas = [c["metadata"] for c in all_chunks]
    
    embs = embedder.encode(texts, normalize_embeddings=True).astype(np.float32)
    index = faiss.IndexFlatIP(embed_dim)
    index.add(embs)
    
    faiss_indexes[c_size] = index
    faiss_stores[c_size] = {"chunks": texts, "metas": metas}
    print(f"FAISS Index for chunk size {c_size} built with {index.ntotal} vectors.")


FAISS Index for chunk size 200 built with 71 vectors.
FAISS Index for chunk size 400 built with 32 vectors.
FAISS Index for chunk size 800 built with 17 vectors.


## 11. Semantic Search Comparison (ChromaDB vs FAISS)

We execute a sample query across both vector engines and compare the retrieved sources and similarity scores.


In [8]:
test_query = "What is the process for submitting an internship task?"
q_emb = embedder.encode([test_query], normalize_embeddings=True)

# Query ChromaDB (size 400)
c_res = chroma_collections[400].query(query_embeddings=q_emb.tolist(), n_results=2)
print("=== ChromaDB Retrieval (Top-2 Chunks) ===")
for i in range(len(c_res["documents"][0])):
    print(f"Rank {i+1} | Source: {c_res['metadatas'][0][i]['source']} | Cosine Distance: {c_res['distances'][0][i]:.4f}")
    print(f"Snippet: {c_res['documents'][0][i][:120]}...\n")

# Query FAISS (size 400)
f_sims, f_idxs = faiss_indexes[400].search(q_emb.astype(np.float32), 2)
print("=== FAISS Retrieval (Top-2 Chunks) ===")
for i in range(2):
    idx = f_idxs[0][i]
    meta = faiss_stores[400]["metas"][idx]
    chunk_txt = faiss_stores[400]["chunks"][idx]
    print(f"Rank {i+1} | Source: {meta['source']} | Cosine Similarity: {f_sims[0][i]:.4f}")
    print(f"Snippet: {chunk_txt[:120]}...\n")


=== ChromaDB Retrieval (Top-2 Chunks) ===
Rank 1 | Source: submission_guidelines.txt | Cosine Distance: 0.4291
Snippet: -16): add basic rag pipeline). Changes must only be pushed to remote repositories after explicit mentor approval or task...

Rank 2 | Source: onboarding.txt | Cosine Distance: 0.4804
Snippet: nel where task briefs are posted. Important company announcements, meeting schedules, and feedback are shared through of...

=== FAISS Retrieval (Top-2 Chunks) ===
Rank 1 | Source: submission_guidelines.txt | Cosine Similarity: 0.5709
Snippet: -16): add basic rag pipeline). Changes must only be pushed to remote repositories after explicit mentor approval or task...

Rank 2 | Source: onboarding.txt | Cosine Similarity: 0.5196
Snippet: nel where task briefs are posted. Important company announcements, meeting schedules, and feedback are shared through of...



## 12. Local LLM Integration

We load the `google-t5/t5-small` model (60M parameters) locally using Hugging Face Transformers. The model generates answers using the retrieved context provided in the prompt.


In [9]:
gen_model_name = "t5-small"
print(f"Loading local generator model: {gen_model_name}...")
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)
print("Local generator initialized successfully.")

def generate_grounded_answer(query, context):
    prompt = f"question: {query} context: {context}"
    inputs = gen_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = gen_model.generate(**inputs, max_new_tokens=60, num_beams=2, early_stopping=True)
    ans = gen_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    return ans if ans else "Information not specified in the provided documentation."

# Test single generation
context_snippet = c_res["documents"][0][0]
sample_answer = generate_grounded_answer(test_query, context_snippet)
print(f"Question: {test_query}")
print(f"Context Snippet: {context_snippet[:150]}...")
print(f"Generated Context-Grounded Answer: '{sample_answer}'")


Loading local generator model: t5-small...
Local generator initialized successfully.
Question: What is the process for submitting an internship task?
Context Snippet: -16): add basic rag pipeline). Changes must only be pushed to remote repositories after explicit mentor approval or task sign-off.

4. Daily Internshi...
Generated Context-Grounded Answer: 'task verification'


## 13. Complete RAG Pipeline

We encapsulate the complete workflow into a modular function:
`query` → `query embedding` → `ChromaDB vector search` → `context assembly` → `T5 generation` → `Context-Grounded Final Answer`.


In [10]:
def run_rag_pipeline(query, chunk_size=400, top_k=2):
    col = chroma_collections[chunk_size]
    q_emb = embedder.encode([query], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q_emb, n_results=top_k)
    
    retrieved_docs = res["documents"][0]
    retrieved_metas = res["metadatas"][0]
    context = " ".join(retrieved_docs)
    
    answer = generate_grounded_answer(query, context)
    return {
        "query": query,
        "chunk_size": chunk_size,
        "retrieved_docs": retrieved_docs,
        "retrieved_metas": retrieved_metas,
        "context": context,
        "answer": answer
    }

# Verify with Question 1
demo_res = run_rag_pipeline("What is the process for submitting an internship task?", chunk_size=400)
print(f"Query: {demo_res['query']}")
print(f"Retrieved Source: {demo_res['retrieved_metas'][0]['source']}")
print(f"Answer: {demo_res['answer']}")


Query: What is the process for submitting an internship task?
Retrieved Source: submission_guidelines.txt
Answer: task verification


## 14. Chunk Size Experiment

### Critical Experimental Requirement:
We evaluate the five benchmark internship questions across all three chunk sizes (200, 400, and 800 characters) to analyze how chunk granularity impacts retrieval accuracy and response generation.

**Benchmark Questions:**
1. What is the process for submitting an internship task?
2. What happens when an intern takes leave?
3. What are the main steps in the training workflow?
4. What should an intern complete before submitting a project?
5. What are the basic onboarding requirements?


In [11]:
questions = [
    "What is the process for submitting an internship task?",
    "What happens when an intern takes leave?",
    "What are the main steps in the training workflow?",
    "What should an intern complete before submitting a project?",
    "What are the basic onboarding requirements?"
]

experiment_results = []

for c_size in [200, 400, 800]:
    for q in questions:
        res = run_rag_pipeline(q, chunk_size=c_size, top_k=2)
        experiment_results.append({
            "Question": q,
            "Chunk Size": c_size,
            "Top Document": res["retrieved_metas"][0]["source"],
            "Retrieved Docs": res["retrieved_docs"],
            "Retrieved Metas": res["retrieved_metas"],
            "Retrieved Text": res["context"],
            "Generated Answer": res["answer"]
        })

exp_df = pd.DataFrame(experiment_results)
print(f"Experiment execution complete: Total {len(exp_df)} evaluations logged.")


Experiment execution complete: Total 15 evaluations logged.


## 15. Response Evaluation (Rule-Based Factual Rubric)

We evaluate responses using a rule-based evaluation rubric derived from known facts in the synthetic demonstration corpus:
- **Relevance (1–5):** Measures alignment of retrieved context & primary authority source with query intent keywords.
- **Correctness against source (1–5):** Verifies factual alignment with source documentation facts without contradictions.
- **Completeness (1–5):** Percentage of expected multi-step procedural facts covered in the retrieved context.
- **Context Grounding (1–5):** Percentage of generated answer content tokens derived directly from context.
- **Overall Score:** Arithmetic mean of the four dimensions.


In [12]:
BENCHMARK_CRITERIA = {
    "What is the process for submitting an internship task?": {
        "primary_doc": "submission_guidelines.txt",
        "query_keywords": ["submit", "submission", "deadline", "task", "deliverables", "tracker", "6:00", "git", "review"],
        "expected_facts": [
            ["6:00 pm", "deadline", "evening"],
            ["deliverables", "notebook", "script", "readme", "outputs", "screenshots"],
            ["git", "linkific-tasks", "ai-ml-internship", "commit", "push"],
            ["tracker", "excel", "sheets", "log", "rebecca", "suyash"]
        ]
    },
    "What happens when an intern takes leave?": {
        "primary_doc": "leave_policy.txt",
        "query_keywords": ["leave", "absence", "mentor", "notice", "hours", "medical", "attendance", "catch-up", "modules"],
        "expected_facts": [
            ["planned leave", "24 hours", "advance", "email", "mentor", "rebecca"],
            ["medical", "emergency", "10:00 am", "certificate"],
            ["responsible", "missed", "technical modules", "complete"],
            ["compensatory", "weekend", "catch-up", "attendance"]
        ]
    },
    "What are the main steps in the training workflow?": {
        "primary_doc": "project_workflow.txt",
        "query_keywords": ["workflow", "lifecycle", "stages", "steps", "training", "requirements", "model", "evaluation"],
        "expected_facts": [
            ["problem definition", "requirements analysis", "stage 1"],
            ["data preprocessing", "validation", "clean", "missing values", "stage 2"],
            ["model development", "training", "baseline", "stage 3"],
            ["evaluation", "metrics", "confusion", "stage 4"],
            ["documentation", "delivery", "notebook", "readme", "stage 5"]
        ]
    },
    "What should an intern complete before submitting a project?": {
        "primary_doc": "project_workflow.txt",
        "query_keywords": ["pre-submission", "checklist", "validation", "complete", "submitting", "errors", "plots", "documentation", "keys"],
        "expected_facts": [
            ["execute", "all cells", "zero runtime errors", "top to bottom"],
            ["confirm", "plots", "charts", "output", "saved"],
            ["sensitive", "api keys", "passwords", "confidential", "security"],
            ["documentation", "accurately reflects", "experimental results"],
            ["sync", "mirror", "repositories", "before committing"]
        ]
    },
    "What are the basic onboarding requirements?": {
        "primary_doc": "onboarding.txt",
        "query_keywords": ["onboarding", "checklist", "requirements", "orientation", "setup", "git", "python", "confidentiality"],
        "expected_facts": [
            ["digital identity", "acceptance verification"],
            ["workspace", "environment", "python", "vscode", "jupyter", "24 hours"],
            ["git credentials", "name", "email", "clone"],
            ["introductory python", "test", "verification"],
            ["confidentiality", "security rules", "acknowledge"]
        ]
    }
}

def score_entry_factual(row):
    q = row["Question"]
    crit = BENCHMARK_CRITERIA[q]
    combined_ctx = row["Retrieved Text"].lower()
    ans_clean = row["Generated Answer"].strip().lower()
    top_source = row["Top Document"]

    # 1. Relevance
    source_match = 1.5 if top_source == crit["primary_doc"] else 0.5
    q_keys = crit["query_keywords"]
    matched_q_keys = sum(1 for k in q_keys if k.lower() in combined_ctx)
    kw_ratio = matched_q_keys / len(q_keys)
    relevance = round(min(5.0, max(1.0, 1.0 + source_match + (2.5 * kw_ratio))), 2)

    # 2. Correctness
    if not ans_clean or ans_clean == "information not specified in the provided documentation." or len(ans_clean) < 3:
        correctness = 1.0
    else:
        fact_hits = 0
        for fact_group in crit["expected_facts"]:
            if any(term in ans_clean for term in fact_group) or any(term in combined_ctx and any(w in ans_clean for w in term.split()) for term in fact_group):
                fact_hits += 1

        ans_tokens = [w for w in ans_clean.split() if len(w) > 3]
        if ans_tokens:
            token_valid = sum(1 for w in ans_tokens if w in combined_ctx) / len(ans_tokens)
        else:
            token_valid = 1.0

        correctness = round(min(5.0, max(1.0, 2.0 + (1.5 * (fact_hits > 0)) + (1.5 * token_valid))), 2)

    # 3. Completeness
    covered_facts = 0
    total_facts = len(crit["expected_facts"])
    for fact_group in crit["expected_facts"]:
        if any(term in combined_ctx for term in fact_group):
            covered_facts += 1
    completeness = round(1.0 + 4.0 * (covered_facts / total_facts), 2)

    # 4. Grounding
    ans_tokens = [w for w in ans_clean.split() if len(w) > 3]
    if not ans_tokens:
        grounding = 5.0 if ans_clean else 1.0
    else:
        matched_tokens = sum(1 for w in ans_tokens if w in combined_ctx)
        grounding_ratio = matched_tokens / len(ans_tokens)
        grounding = round(min(5.0, max(1.0, 1.0 + 4.0 * grounding_ratio)), 2)

    overall = round((relevance + correctness + completeness + grounding) / 4.0, 2)
    return pd.Series([relevance, correctness, completeness, grounding, overall],
                     index=["Relevance", "Correctness", "Completeness", "Grounding", "Overall Score"])

eval_scores = exp_df.apply(score_entry_factual, axis=1)
full_eval_df = pd.concat([exp_df[["Question", "Chunk Size", "Top Document", "Generated Answer"]], eval_scores], axis=1)
display(full_eval_df)


                                                       Question  Chunk Size               Top Document                                                                                                                             Generated Answer  Relevance  Correctness  Completeness  Grounding  Overall Score
0        What is the process for submitting an internship task?         200  submission_guidelines.txt                                                                                                                                  6:00 PM IST       4.44          5.0           3.0        5.0           4.36
1                      What happens when an intern takes leave?         200           leave_policy.txt                                                                          they remain responsible for completing all missed technical modules       3.33          5.0           3.0        5.0           4.08
2             What are the main steps in the training workflow?         200 

## 16. Comparison Results

We aggregate the empirical evaluation scores by chunk size and visualize comparative performance.


In [13]:
summary_table = full_eval_df.groupby("Chunk Size")[["Relevance", "Correctness", "Completeness", "Grounding", "Overall Score"]].mean().reset_index()
display(summary_table)

# Plot performance comparison
fig, ax = plt.subplots(figsize=(8, 4.5))
bar_width = 0.35
x = np.arange(len(summary_table))

b1 = ax.bar(x - bar_width/2, summary_table["Overall Score"], bar_width, label="Average Overall Score", color="#2b5c8f")
b2 = ax.bar(x + bar_width/2, summary_table["Completeness"], bar_width, label="Completeness Score", color="#4fa3a5")

ax.set_xlabel("Chunk Size (Characters)", fontweight="bold")
ax.set_ylabel("Score (1 to 5 Scale)", fontweight="bold")
ax.set_title("RAG Response Quality by Chunk Size (Empirical Evaluation)", fontweight="bold", pad=12)
ax.set_xticks(x)
ax.set_xticklabels([f"Size {int(s)}" for s in summary_table["Chunk Size"]])
ax.set_ylim(0, 5.5)
ax.legend(loc="upper left")

for bar in b1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, h + 0.1, f"{h:.2f}", ha="center", va="bottom", fontweight="bold", fontsize=9)
for bar in b2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, h + 0.1, f"{h:.2f}", ha="center", va="bottom", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()


   Chunk Size  Relevance  Correctness  Completeness  Grounding  Overall Score
0         200      3.568          4.7          2.92        5.0          4.048
1         400      4.054          4.4          4.48        5.0          4.482
2         800      4.338          4.7          4.48        5.0          4.628


## 17. Best Performing Chunk Size

Based strictly on empirical evaluation across all 5 benchmark questions:

### Winner: **Chunk Size 800 characters** (Overall Score: **4.63 / 5.00**)

### Detailed Findings Explanation:
1. **Chunk Size 200 (Overall: 4.05, Completeness: 2.92):**
   - 200 characters corresponds to only ~25–35 words.
   - Procedural policy questions (e.g., submission deliverables checklists and onboarding steps) require multi-sentence context. Slicing at 200 characters split sentences mid-procedure, causing the lowest completeness score (2.92).
2. **Chunk Size 400 (Overall: 4.48, Completeness: 4.48):**
   - 400 characters corresponds to ~55–80 words, roughly the length of an individual operational policy clause.
   - It significantly improved completeness over 200 characters while maintaining strong query alignment.
3. **Chunk Size 800 (Overall: 4.63, Relevance: 4.34, Completeness: 4.48):**
   - 800 characters encompasses 120–160 words.
   - In our demonstration corpus, 800 characters provided sufficient context window to preserve complete multi-step instructions and achieve the highest relevance (4.34) without truncating necessary guidelines.


## 18. Observations

Key technical insights derived from the Day 16 RAG experiments:
1. **Pretrained Model Acceleration:** Pretrained models reduce the need to train a model from scratch and can accelerate prototyping and development.
2. **Semantic Matching Efficacy:** Dense vector search successfully resolved queries that did not share literal keywords with the source text (e.g., mapping *"takes leave"* to *"leave policy and attendance procedures"*).
3. **Chunk Size Trade-Offs:** There is an inherent trade-off between chunk specificity and context completeness. Smaller chunks improve retrieval specificity but risk fragmenting meaning; larger chunks improve context richness but may introduce broader scope.
4. **Grounding Importance:** In this experiment, providing explicit retrieved context helped keep the generated responses aligned with the demonstration documentation, illustrating the value of RAG for domain-specific question answering.
5. **Vector DB Usability:** ChromaDB provided seamless collection management and metadata retrieval out-of-the-box, while FAISS offered optimized low-level vector similarity search.


## 19. Limitations

- **Synthetic Demonstration Corpus:** Experiments used a synthetic 5-document demonstration dataset (~10.6K characters) to maintain confidentiality; production enterprise systems index thousands of multi-page PDFs.
- **Fixed-Character Chunking:** Character-window slicing does not account for document markdown headers or natural paragraph boundaries (semantic chunking).
- **Lightweight Generator Capacity:** `t5-small` (60M parameters) produces brief answers; larger models (e.g., LLaMA-3, Mistral, Flan-T5) would synthesize more nuanced multi-paragraph explanations.
- **Single-Turn Evaluation:** The pipeline evaluates single-turn Q&A without conversation history memory.


## 20. Conclusion

For this demonstration dataset and evaluation set, the **800-character chunk size** provided the best observed balance between retrieval relevance and response completeness.

By combining character chunking, dense vector embeddings via `all-MiniLM-L6-v2`, vector indexing with ChromaDB and FAISS, and grounded sequence-to-sequence answer generation, we built a fully functional RAG pipeline. Day 16 established a solid, verified foundation for domain-specific AI applications.
